In [ ]:
# ===== 8a: Paper 2 detector baseline (self-contained) =====
import os, numpy as np, pandas as pd, torch, torch.nn as nn, json
from google.colab import drive; drive.mount("/content/drive")
from sklearn.metrics import roc_auc_score
P = "/content/drive/MyDrive/AI-DTSec/data/wustl_ehms/wustl-ehms-2020_with_attacks_categories.csv"
OUT = "/content/drive/MyDrive/AI-DTSec/results/redteam"; os.makedirs(OUT, exist_ok=True)
df = pd.read_csv(P); df.columns = [c.strip() for c in df.columns]
y = df["Label"].astype(int).values; N = len(y)
cat = df["Attack Category"].astype(str).str.lower().values
ids = ["Dir","Flgs","SrcAddr","DstAddr","Sport","Dport","SrcMac","DstMac","Packet_num"]
cols = [c for c in df.columns if c not in ids+["Label","Attack Category"]]
X = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0).values.astype(np.float32)
print("features:", len(cols), "| attack categories:", pd.Series(cat).value_counts().to_dict())

# leakage-aware split: 20 contiguous blocks, test = blocks 3,7,11,15,19, purge 100 samples around test blocks
edges = np.linspace(0, N, 21).astype(int); test_blocks = [3, 7, 11, 15, 19]
te = np.concatenate([np.arange(edges[b], edges[b+1]) for b in test_blocks])
banned = np.zeros(N, bool)
for b in test_blocks: banned[max(0, edges[b]-100):min(N, edges[b+1]+100)] = True
tr = np.where(~banned)[0]
trn = tr[y[tr] == 0]                     # detector sees normal training samples only
print(f"train samples {len(tr)} (normal used {len(trn)}) | test {len(te)} (attacks {int(y[te].sum())})")

mu, sd = X[trn].mean(0), X[trn].std(0) + 1e-6
Z = (X - mu) / sd
torch.manual_seed(0); dev = "cuda" if torch.cuda.is_available() else "cpu"
ae = nn.Sequential(nn.Linear(34,24), nn.ReLU(), nn.Linear(24,12), nn.ReLU(), nn.Linear(12,6),
                   nn.ReLU(), nn.Linear(6,12), nn.ReLU(), nn.Linear(12,24), nn.ReLU(), nn.Linear(24,34)).to(dev)
A = torch.tensor(Z[trn], device=dev); opt = torch.optim.Adam(ae.parameters(), 1e-3)
for ep in range(80):
    perm = torch.randperm(len(A), device=dev)
    for i in range(0, len(A), 128):
        idx = perm[i:i+128]; opt.zero_grad(); nn.functional.mse_loss(ae(A[idx]), A[idx]).backward(); opt.step()
def score(Zm):
    with torch.no_grad():
        t = torch.tensor(Zm, dtype=torch.float32, device=dev); return ((ae(t) - t)**2).mean(1).cpu().numpy()
thr = float(np.percentile(score(Z[trn]), 95))        # 95th percentile of normal training error
s = score(Z[te]); yt = y[te]; ct = cat[te]
print(f"\nthreshold (95th pct, normal train) = {thr:.4f}")
print(f"hold-out AUC-ROC = {roc_auc_score(yt, s):.3f} | false-positive rate on normal = {(s[yt==0] > thr).mean():.3f}")
for c in sorted(set(ct)):
    if c in ("normal", "nan"): continue
    m = ct == c
    print(f"  {c:18s} n={m.sum():4d} | detected (recall) = {(s[m] > thr).mean():.3f} | AUC vs normal = {roc_auc_score(np.r_[np.zeros((yt==0).sum()), np.ones(m.sum())], np.r_[s[yt==0], s[m]]):.3f}")
# save artifacts for 8b
torch.save(ae.state_dict(), f"{OUT}/ae_state.pt")
np.savez(f"{OUT}/baseline.npz", mu=mu, sd=sd, thr=thr, te=te, tr=tr, cols=np.array(cols))
json.dump({"thr": thr}, open(f"{OUT}/baseline.json", "w"))
print("saved to", OUT)

Mounted at /content/drive
features: 34 | attack categories: {'normal': 14272, 'spoofing': 1124, 'data alteration': 922}
train samples 11338 (normal used 9866) | test 4080 (attacks 452)

threshold (95th pct, normal train) = 0.2210
hold-out AUC-ROC = 0.750 | false-positive rate on normal = 0.053
  data alteration    n= 264 | detected (recall) = 0.830 | AUC vs normal = 0.965
  spoofing           n= 188 | detected (recall) = 0.027 | AUC vs normal = 0.446
saved to /content/drive/MyDrive/AI-DTSec/results/redteam


Section 8b: the red-team agent on the twin.

In [ ]:
# ===== 8b: red-team evasion on the twin (data-alteration attacks), sequential vectorized =====
import numpy as np, pandas as pd, torch, torch.nn as nn, json, time
from google.colab import drive; drive.mount("/content/drive")
P = "/content/drive/MyDrive/AI-DTSec/data/wustl_ehms/wustl-ehms-2020_with_attacks_categories.csv"
OUT = "/content/drive/MyDrive/AI-DTSec/results/redteam"
dev = "cuda" if torch.cuda.is_available() else "cpu"
b = np.load(f"{OUT}/baseline.npz", allow_pickle=True)
mu, sd, thr, te, tr, cols = b["mu"], b["sd"], float(b["thr"]), b["te"], b["tr"], list(b["cols"])
df = pd.read_csv(P); df.columns = [c.strip() for c in df.columns]
y = df["Label"].astype(int).values; cat = df["Attack Category"].astype(str).str.lower().values
X = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0).values.astype(np.float32)
Z = ((X - mu) / sd).astype(np.float32)
ae = nn.Sequential(nn.Linear(34,24), nn.ReLU(), nn.Linear(24,12), nn.ReLU(), nn.Linear(12,6),
                   nn.ReLU(), nn.Linear(6,12), nn.ReLU(), nn.Linear(12,24), nn.ReLU(), nn.Linear(24,34)).to(dev)
ae.load_state_dict(torch.load(f"{OUT}/ae_state.pt", map_location=dev)); ae.eval()
@torch.no_grad()
def score(t): return ((ae(t) - t)**2).mean(-1)

trn = tr[y[tr] == 0]
lo = torch.tensor(np.percentile(Z[trn], 0.5, axis=0), device=dev); hi = torch.tensor(np.percentile(Z[trn], 99.5, axis=0), device=dev)
bio = ["Temp","SpO2","Pulse_Rate","SYS","DIA","Heart_rate","Resp_Rate","ST"]
groups = {"biometric (8)": np.array([c in bio for c in cols]),
          "network (26)": np.array([c not in bio for c in cols]),
          "all (34)": np.ones(len(cols), bool)}

S_idx = te[cat[te] == "data alteration"]
x0 = torch.tensor(Z[S_idx], device=dev); s0 = score(x0)
det0 = s0 > thr
print(f"data-alteration test samples: {len(S_idx)} | detected before attack: {int(det0.sum())} ({det0.float().mean():.3f})")
lo_s = torch.minimum(lo, x0); hi_s = torch.maximum(hi, x0)       # stay in normal range, or original value if already outside

def attack(mask, eps, method, budget_gen=25, pop=16, seed=0):
    g = torch.Generator(device=dev).manual_seed(seed)
    m = torch.tensor(mask, device=dev, dtype=torch.float32)
    S, D = x0.shape
    best_x, best_s = x0.clone(), s0.clone()
    for gen in range(budget_gen):
        if method == "random":
            d = (torch.rand(S, pop, D, device=dev, generator=g) * 2 - 1) * eps
            cand = x0[:, None, :] + d * m
        else:                                                    # guided (1+lambda)-ES around the current best
            d = torch.randn(S, pop, D, device=dev, generator=g) * (eps / 2)
            cand = best_x[:, None, :] + d * m
            cand = x0[:, None, :] + (cand - x0[:, None, :]).clamp(-eps, eps) * m
        cand = torch.minimum(torch.maximum(cand, lo_s[:, None, :]), hi_s[:, None, :])
        sc = score(cand.reshape(-1, D)).reshape(S, pop)
        mn, ai = sc.min(1)
        upd = mn < best_s
        best_s = torch.where(upd, mn, best_s)
        best_x = torch.where(upd[:, None], cand[torch.arange(S), ai], best_x)
    return best_s

rows = []
for gname, mask in groups.items():
    for eps in [0.1, 0.25, 0.5, 1.0, 2.0]:
        for method in ["random", "guided"]:
            t0 = time.time(); s1 = attack(mask, eps, method); dt = time.time() - t0
            evaded = (det0 & (s1 <= thr)).sum().item() / max(int(det0.sum()), 1)
            rows.append(dict(group=gname, eps=eps, method=method, detected_after=(s1 > thr).float().mean().item(),
                             evasion_rate=evaded, seconds=round(dt, 2)))
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/redteam_8b.csv", index=False)
print("\nEVASION RATE (share of originally detected alterations that fall below the threshold)")
print(res.pivot_table(index=["group", "eps"], columns="method", values="evasion_rate").round(3).to_string())
print("\nRECALL AFTER ATTACK (all alteration samples; before attack = %.3f)" % det0.float().mean().item())
print(res.pivot_table(index=["group", "eps"], columns="method", values="detected_after").round(3).to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data-alteration test samples: 264 | detected before attack: 219 (0.830)

EVASION RATE (share of originally detected alterations that fall below the threshold)
method              guided  random
group         eps                 
all (34)      0.10   0.201   0.119
              0.25   0.447   0.292
              0.50   0.676   0.489
              1.00   0.858   0.689
              2.00   0.922   0.817
biometric (8) 0.10   0.073   0.059
              0.25   0.155   0.114
              0.50   0.324   0.205
              1.00   0.502   0.361
              2.00   0.731   0.589
network (26)  0.10   0.114   0.096
              0.25   0.315   0.251
              0.50   0.489   0.406
              1.00   0.685   0.584
              2.00   0.749   0.699

RECALL AFTER ATTACK (all alteration samples; before attack = 0.830)
method              guided  random
group        

Section 8c: close the loop (attack, retrain, re-attack).

In [ ]:
# ===== 8c: red-team loop: attack -> adversarial retraining -> re-attack (self-contained) =====
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from google.colab import drive; drive.mount("/content/drive")
from sklearn.metrics import roc_auc_score
P = "/content/drive/MyDrive/AI-DTSec/data/wustl_ehms/wustl-ehms-2020_with_attacks_categories.csv"
OUT = "/content/drive/MyDrive/AI-DTSec/results/redteam"
dev = "cuda" if torch.cuda.is_available() else "cpu"
b = np.load(f"{OUT}/baseline.npz", allow_pickle=True)
mu, sd, te, tr, cols = b["mu"], b["sd"], b["te"], b["tr"], list(b["cols"])
df = pd.read_csv(P); df.columns = [c.strip() for c in df.columns]
y = df["Label"].astype(int).values; cat = df["Attack Category"].astype(str).str.lower().values
X = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0).values.astype(np.float32)
Z = ((X - mu) / sd).astype(np.float32)
trn = tr[y[tr] == 0]
lo = torch.tensor(np.percentile(Z[trn], 0.5, axis=0), device=dev); hi = torch.tensor(np.percentile(Z[trn], 99.5, axis=0), device=dev)

def train_clf(Xt, yt, seed=0, epochs=40):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(34,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1)).to(dev)
    A = torch.tensor(Xt, device=dev); t = torch.tensor(yt, dtype=torch.float32, device=dev)
    pw = torch.tensor((1 - t.mean()) / t.mean(), device=dev); opt = torch.optim.Adam(m.parameters(), 1e-3)
    for ep in range(epochs):
        perm = torch.randperm(len(A), device=dev)
        for i in range(0, len(A), 256):
            idx = perm[i:i+256]; opt.zero_grad()
            nn.functional.binary_cross_entropy_with_logits(m(A[idx]).squeeze(1), t[idx], pos_weight=pw).backward(); opt.step()
    m.eval()
    return lambda T: torch.sigmoid(m(T).squeeze(-1)).detach()   # suspicion score in [0,1]

def attack(score_fn, x0, eps, seed, gens=25, pop=16):
    g = torch.Generator(device=dev).manual_seed(seed); S, D = x0.shape
    lo_s = torch.minimum(lo, x0); hi_s = torch.maximum(hi, x0)
    best_x, best_s = x0.clone(), score_fn(x0)
    for _ in range(gens):                                        # guided (1+lambda)-ES, all 34 features mutable
        cand = best_x[:, None, :] + torch.randn(S, pop, D, device=dev, generator=g) * (eps / 2)
        cand = x0[:, None, :] + (cand - x0[:, None, :]).clamp(-eps, eps)
        cand = torch.minimum(torch.maximum(cand, lo_s[:, None, :]), hi_s[:, None, :])
        sc = score_fn(cand.reshape(-1, D)).reshape(S, pop); mn, ai = sc.min(1); upd = mn < best_s
        best_s = torch.where(upd, mn, best_s); best_x = torch.where(upd[:, None], cand[torch.arange(S), ai], best_x)
    return best_s, best_x

Zt = torch.tensor(Z, device=dev); THR = 0.5
alt_te = te[cat[te] == "data alteration"]; spo_te = te[cat[te] == "spoofing"]; nor_te = te[y[te] == 0]
def evaluate(name, fn, seed):
    s = fn(Zt[te]).cpu().numpy()
    out = dict(model=name, AUC=roc_auc_score(y[te], s), FPR=float((fn(Zt[nor_te]) > THR).float().mean()),
               recall_alteration=float((fn(Zt[alt_te]) > THR).float().mean()), recall_spoofing=float((fn(Zt[spo_te]) > THR).float().mean()))
    x0 = Zt[alt_te]; det0 = fn(x0) > THR
    for eps in [0.25, 0.5, 1.0]:
        s1, _ = attack(fn, x0, eps, seed)
        out[f"evasion@{eps}"] = float((det0 & (s1 <= THR)).sum() / max(int(det0.sum()), 1))
    return out

rows = []
C0 = train_clf(Z[tr], y[tr], seed=0); rows.append(evaluate("supervised (clean training)", C0, seed=1))
print(rows[-1], flush=True)

# red team on the TRAINING blocks: craft evasive variants of training alterations against C0, add as attack-labeled samples
alt_tr = tr[cat[tr] == "data alteration"]; adv = []
for k, eps in enumerate([0.25, 0.5, 1.0]):
    _, bx = attack(C0, Zt[alt_tr], eps, seed=100 + k); adv.append(bx.cpu().numpy())
adv = np.concatenate(adv)
Xaug = np.vstack([Z[tr], adv]); yaug = np.r_[y[tr], np.ones(len(adv), dtype=y.dtype)]
print(f"adversarial retraining: +{len(adv)} red-team variants (training blocks only)")
C1 = train_clf(Xaug, yaug, seed=0); rows.append(evaluate("supervised (after 1 red-team round)", C1, seed=2))
print(rows[-1], flush=True)
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/redteam_8c.csv", index=False)
print(); print(res.round(3).T.to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_2243/154560084.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pw = torch.tensor((1 - t.mean()) / t.mean(), device=dev); opt = torch.optim.Adam(m.parameters(), 1e-3)


{'model': 'supervised (clean training)', 'AUC': np.float64(0.8511021699466293), 'FPR': 0.2409040927886963, 'recall_alteration': 0.9962121248245239, 'recall_spoofing': 0.3617021143436432, 'evasion@0.25': 0.03041825070977211, 'evasion@0.5': 0.3726235628128052, 'evasion@1.0': 0.8973383903503418}
adversarial retraining: +1881 red-team variants (training blocks only)


/tmp/ipykernel_2243/154560084.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pw = torch.tensor((1 - t.mean()) / t.mean(), device=dev); opt = torch.optim.Adam(m.parameters(), 1e-3)


{'model': 'supervised (after 1 red-team round)', 'AUC': np.float64(0.8437118868973862), 'FPR': 0.026736494153738022, 'recall_alteration': 0.9962121248245239, 'recall_spoofing': 0.021276595070958138, 'evasion@0.25': 0.0076045626774430275, 'evasion@0.5': 0.13688212633132935, 'evasion@1.0': 0.8897338509559631}

                                             0                                    1
model              supervised (clean training)  supervised (after 1 red-team round)
AUC                                      0.851                                0.844
FPR                                      0.241                                0.027
recall_alteration                        0.996                                0.996
recall_spoofing                          0.362                                0.021
evasion@0.25                              0.03                                0.008
evasion@0.5                              0.373                                0.137
evasion@1.0       

Section 8d: the same loop at matched operating points, with repeats.

In [ ]:
# ===== 8d: red-team loop at matched FPR (5%), 3 seeds, 2 rounds (self-contained) =====
import numpy as np, pandas as pd, torch, torch.nn as nn
from google.colab import drive; drive.mount("/content/drive")
from sklearn.metrics import roc_auc_score
P = "/content/drive/MyDrive/AI-DTSec/data/wustl_ehms/wustl-ehms-2020_with_attacks_categories.csv"
OUT = "/content/drive/MyDrive/AI-DTSec/results/redteam"
dev = "cuda" if torch.cuda.is_available() else "cpu"
b = np.load(f"{OUT}/baseline.npz", allow_pickle=True)
mu, sd, te, tr, cols = b["mu"], b["sd"], b["te"], b["tr"], list(b["cols"])
df = pd.read_csv(P); df.columns = [c.strip() for c in df.columns]
y = df["Label"].astype(int).values; cat = df["Attack Category"].astype(str).str.lower().values
X = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0).values.astype(np.float32)
Z = ((X - mu) / sd).astype(np.float32); Zt = torch.tensor(Z, device=dev)

# validation slice = last 20% (by time) of the training indices; excluded from model fitting; used only to set thresholds
tr = np.sort(tr); nval = len(tr) // 5
val, fit = tr[-nval:], tr[:-nval]
fit = fit[fit < val.min() - 100]                      # purge between fit and validation
trn = fit[y[fit] == 0]
lo = torch.tensor(np.percentile(Z[trn], 0.5, axis=0), device=dev); hi = torch.tensor(np.percentile(Z[trn], 99.5, axis=0), device=dev)
PW = float((1 - y[fit].mean()) / y[fit].mean())       # class weight fixed from clean data
print(f"fit {len(fit)} | validation {len(val)} (normal {int((y[val]==0).sum())}) | pos_weight {PW:.2f}")

def train_clf(Xt, yt, seed, epochs=40):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(34,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1)).to(dev)
    A = torch.tensor(Xt, device=dev); t = torch.tensor(yt, dtype=torch.float32, device=dev)
    opt = torch.optim.Adam(m.parameters(), 1e-3); pw = torch.tensor(PW, device=dev)
    for ep in range(epochs):
        perm = torch.randperm(len(A), device=dev)
        for i in range(0, len(A), 256):
            idx = perm[i:i+256]; opt.zero_grad()
            nn.functional.binary_cross_entropy_with_logits(m(A[idx]).squeeze(1), t[idx], pos_weight=pw).backward(); opt.step()
    m.eval()
    return lambda T: torch.sigmoid(m(T).squeeze(-1)).detach()

def attack(fn, x0, eps, seed, gens=25, pop=16):
    g = torch.Generator(device=dev).manual_seed(seed); S, D = x0.shape
    lo_s = torch.minimum(lo, x0); hi_s = torch.maximum(hi, x0); best_x, best_s = x0.clone(), fn(x0)
    for _ in range(gens):
        cand = best_x[:, None, :] + torch.randn(S, pop, D, device=dev, generator=g) * (eps / 2)
        cand = x0[:, None, :] + (cand - x0[:, None, :]).clamp(-eps, eps)
        cand = torch.minimum(torch.maximum(cand, lo_s[:, None, :]), hi_s[:, None, :])
        sc = fn(cand.reshape(-1, D)).reshape(S, pop); mn, ai = sc.min(1); upd = mn < best_s
        best_s = torch.where(upd, mn, best_s); best_x = torch.where(upd[:, None], cand[torch.arange(S), ai], best_x)
    return best_s, best_x

alt_te = te[cat[te] == "data alteration"]; spo_te = te[cat[te] == "spoofing"]; nor_te = te[y[te] == 0]
alt_fit = fit[cat[fit] == "data alteration"]
def calibrate(fn):                                     # threshold = 95th percentile of normal validation scores -> 5% FPR
    return float(np.percentile(fn(Zt[val[y[val] == 0]]).cpu().numpy(), 95))
def evaluate(fn, thr, seed):
    r = dict(AUC=roc_auc_score(y[te], fn(Zt[te]).cpu().numpy()), FPR=float((fn(Zt[nor_te]) > thr).float().mean()),
             rec_alt=float((fn(Zt[alt_te]) > thr).float().mean()), rec_spoof=float((fn(Zt[spo_te]) > thr).float().mean()))
    x0 = Zt[alt_te]; det0 = fn(x0) > thr
    for eps in [0.25, 0.5, 1.0]:
        s1, _ = attack(fn, x0, eps, seed); r[f"evade@{eps}"] = float((det0 & (s1 <= thr)).sum() / max(int(det0.sum()), 1))
    return r

rows = []
for seed in range(3):
    fn = train_clf(Z[fit], y[fit], seed); thr = calibrate(fn)
    rows.append(dict(model="clean", seed=seed, **evaluate(fn, thr, 1000 + seed)))
    adv = np.concatenate([attack(fn, Zt[alt_fit], e, 200 + seed * 10 + k)[1].cpu().numpy() for k, e in enumerate([0.25, 0.5, 1.0])])
    fn1 = train_clf(np.vstack([Z[fit], adv]), np.r_[y[fit], np.ones(len(adv), dtype=y.dtype)], seed); thr1 = calibrate(fn1)
    rows.append(dict(model="round 1", seed=seed, **evaluate(fn1, thr1, 1000 + seed)))
    adv2 = np.concatenate([attack(fn1, Zt[alt_fit], e, 300 + seed * 10 + k)[1].cpu().numpy() for k, e in enumerate([0.25, 0.5, 1.0])])
    fn2 = train_clf(np.vstack([Z[fit], adv, adv2]), np.r_[y[fit], np.ones(len(adv) + len(adv2), dtype=y.dtype)], seed); thr2 = calibrate(fn2)
    rows.append(dict(model="round 2", seed=seed, **evaluate(fn2, thr2, 1000 + seed)))
    print("seed", seed, "done", flush=True)
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/redteam_8d.csv", index=False)
agg = res.groupby("model").agg(["mean", "std"]).drop(columns="seed", level=0)
print(agg.round(3).T.to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fit 8971 | validation 2267 (normal 1990) | pos_weight 6.51
seed 0 done
seed 1 done
seed 2 done
model            clean  round 1  round 2
AUC        mean  0.852    0.849    0.845
           std   0.002    0.001    0.003
FPR        mean  0.075    0.041    0.098
           std   0.041    0.012    0.038
rec_alt    mean  0.996    0.996    0.994
           std   0.000    0.000    0.004
rec_spoof  mean  0.135    0.105    0.174
           std   0.064    0.020    0.030
evade@0.25 mean  0.023    0.011    0.000
           std   0.030    0.010    0.000
evade@0.5  mean  0.307    0.180    0.001
           std   0.143    0.081    0.002
evade@1.0  mean  0.848    0.825    0.320
           std   0.024    0.025    0.312


Section 8e: parallel speedup of the red-team search


In [ ]:
# ===== 8e: parallel speedup of the red-team scenario grid (self-contained) =====
import os, time, numpy as np, pandas as pd, torch, torch.nn as nn, multiprocessing as mp
from google.colab import drive; drive.mount("/content/drive")
P = "/content/drive/MyDrive/AI-DTSec/data/wustl_ehms/wustl-ehms-2020_with_attacks_categories.csv"
OUT = "/content/drive/MyDrive/AI-DTSec/results/redteam"
b = np.load(f"{OUT}/baseline.npz", allow_pickle=True)
mu, sd, thr, te, tr, cols = b["mu"], b["sd"], float(b["thr"]), b["te"], b["tr"], list(b["cols"])
df = pd.read_csv(P); df.columns = [c.strip() for c in df.columns]
y = df["Label"].astype(int).values; cat = df["Attack Category"].astype(str).str.lower().values
X = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0).values.astype(np.float32)
Z = ((X - mu) / sd).astype(np.float32); trn = tr[y[tr] == 0]
bio = ["Temp","SpO2","Pulse_Rate","SYS","DIA","Heart_rate","Resp_Rate","ST"]
masks = {"biometric": np.array([c in bio for c in cols]), "network": np.array([c not in bio for c in cols]), "all": np.ones(len(cols), bool)}
sd_ae = torch.load(f"{OUT}/ae_state.pt", map_location="cpu")
np.savez("/content/rt_data.npz", x0=Z[te[cat[te] == "data alteration"]], lo=np.percentile(Z[trn], 0.5, axis=0), hi=np.percentile(Z[trn], 99.5, axis=0),
         thr=thr, **{f"mask_{k}": v for k, v in masks.items()})
torch.save(sd_ae, "/content/rt_ae.pt")

open("/content/rt_worker.py", "w").write('''
import numpy as np, torch, torch.nn as nn
torch.set_num_threads(1)
G = {}
def build():
    return nn.Sequential(nn.Linear(34,24), nn.ReLU(), nn.Linear(24,12), nn.ReLU(), nn.Linear(12,6), nn.ReLU(),
                         nn.Linear(6,12), nn.ReLU(), nn.Linear(12,24), nn.ReLU(), nn.Linear(24,34))
def init():
    d = np.load("/content/rt_data.npz"); G["x0"] = torch.tensor(d["x0"]); G["lo"] = torch.tensor(d["lo"]); G["hi"] = torch.tensor(d["hi"])
    G["thr"] = float(d["thr"]); G["masks"] = {k: torch.tensor(d["mask_" + k], dtype=torch.float32) for k in ("biometric", "network", "all")}
    m = build(); m.load_state_dict(torch.load("/content/rt_ae.pt", map_location="cpu")); m.eval(); G["ae"] = m
def run(task, dev=None):
    group, eps, method, seed = task
    m = G["ae"]; x0 = G["x0"]; lo, hi = G["lo"], G["hi"]; mk = G["masks"][group]
    gen = torch.Generator().manual_seed(seed); S, D = x0.shape
    def score(t):
        with torch.no_grad(): return ((m(t) - t) ** 2).mean(-1)
    lo_s = torch.minimum(lo, x0); hi_s = torch.maximum(hi, x0); best_x, best_s = x0.clone(), score(x0); s0 = best_s.clone()
    for _ in range(200):
        if method == "random":
            cand = x0[:, None, :] + (torch.rand(S, 16, D, generator=gen) * 2 - 1) * eps * mk
        else:
            cand = best_x[:, None, :] + torch.randn(S, 16, D, generator=gen) * (eps / 2) * mk
            cand = x0[:, None, :] + (cand - x0[:, None, :]).clamp(-eps, eps) * mk
        cand = torch.minimum(torch.maximum(cand, lo_s[:, None, :]), hi_s[:, None, :])
        sc = score(cand.reshape(-1, D)).reshape(S, 16); mn, ai = sc.min(1); upd = mn < best_s
        best_s = torch.where(upd, mn, best_s); best_x = torch.where(upd[:, None], cand[torch.arange(S), ai], best_x)
    det0 = s0 > G["thr"]
    return task, float((det0 & (best_s <= G["thr"])).sum() / max(int(det0.sum()), 1))
''')
import sys; sys.path.insert(0, "/content"); import rt_worker as W
tasks = [(g, e, m, s) for g in masks for e in [0.1, 0.25, 0.5, 1.0, 2.0] for m in ["random", "guided"] for s in range(3)]
print("scenarios:", len(tasks), "| CPU cores visible:", os.cpu_count())

W.init(); W.run(tasks[0])                                  # warm-up
t0 = time.time(); seq = [W.run(t) for t in tasks]; t_seq = time.time() - t0
print(f"sequential (1 process): {t_seq:.1f} s")

rows = [dict(workers=0, seconds=t_seq, speedup=1.0, efficiency=1.0, note="sequential")]
ctx = mp.get_context("spawn")
for w in [1, 2, 4, 8]:
    if w > os.cpu_count(): continue
    with ctx.Pool(w, initializer=W.init) as pool:
        pool.map(W.run, tasks[:w], chunksize=1)            # warm-up: workers started and loaded
        t0 = time.time(); par = pool.map(W.run, tasks, chunksize=1); dt = time.time() - t0
    same = all(abs(a[1] - b_[1]) < 1e-9 for a, b_ in zip(seq, par))
    rows.append(dict(workers=w, seconds=dt, speedup=t_seq / dt, efficiency=t_seq / dt / w, note=f"identical results: {same}"))
    print(f"{w} workers: {dt:.1f} s | speedup {t_seq/dt:.2f} | efficiency {t_seq/dt/w:.2f} | identical: {same}", flush=True)
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/redteam_8e.csv", index=False); print(); print(res.round(3).to_string(index=False))
print("\nFor reference: the same grid vectorised on one GPU was not timed here; the sequential CPU run above is the baseline.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
scenarios: 90 | CPU cores visible: 2
sequential (1 process): 51.1 s
1 workers: 57.2 s | speedup 0.89 | efficiency 0.89 | identical: True
2 workers: 47.6 s | speedup 1.07 | efficiency 0.54 | identical: True

 workers  seconds  speedup  efficiency                    note
       0   51.127    1.000       1.000              sequential
       1   57.202    0.894       0.894 identical results: True
       2   47.615    1.074       0.537 identical results: True

For reference: the same grid vectorised on one GPU was not timed here; the sequential CPU run above is the baseline.


Diagnostic: does this runtime provide two real cores?

In [ ]:
# ===== 8e-check v2: does this machine scale across processes at all? =====
import os, sys, time, multiprocessing as mp
open("/content/burn_mod.py", "w").write('''
def burn(n):
    s = 0
    for i in range(n): s += i * i % 7
    return s
''')
sys.path.insert(0, "/content"); import burn_mod
N = 6_000_000; ctx = mp.get_context("spawn")
t0 = time.time(); [burn_mod.burn(N) for _ in range(4)]; t1 = time.time() - t0
print("cores visible:", os.cpu_count(), "| 4 jobs sequential:", round(t1, 1), "s", flush=True)
for w in (2, 4):
    with ctx.Pool(w) as p:
        p.map(burn_mod.burn, [1000] * w); t0 = time.time(); p.map(burn_mod.burn, [N] * 4, chunksize=1); dt = time.time() - t0
    print(f"{w} processes: {dt:.1f} s | speedup {t1/dt:.2f}", flush=True)

cores visible: 2 | 4 jobs sequential: 2.8 s
2 processes: 2.9 s | speedup 0.96
4 processes: 2.9 s | speedup 0.97


In [1]:
# ===== 9b: independent attackers vs hardened detectors (self-contained) =====
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from google.colab import drive; drive.mount("/content/drive")
from sklearn.metrics import roc_auc_score
P   = "/content/drive/MyDrive/AI-DTSec/data/wustl_ehms/wustl-ehms-2020_with_attacks_categories.csv"
OUT = "/content/drive/MyDrive/AI-DTSec/results/redteam"
dev = "cuda" if torch.cuda.is_available() else "cpu"
b = np.load(f"{OUT}/baseline.npz", allow_pickle=True)
mu, sd, te, tr, cols = b["mu"], b["sd"], b["te"], b["tr"], list(b["cols"])
df = pd.read_csv(P); df.columns = [c.strip() for c in df.columns]
y = df["Label"].astype(int).values; cat = df["Attack Category"].astype(str).str.lower().values
X = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0).values.astype(np.float32)
Z = ((X - mu) / sd).astype(np.float32); Zt = torch.tensor(Z, device=dev)

# same fit / validation split as 8d
tr = np.sort(tr); nval = len(tr) // 5
val, fit = tr[-nval:], tr[:-nval]; fit = fit[fit < val.min() - 100]
trn = fit[y[fit] == 0]
lo = torch.tensor(np.percentile(Z[trn], 0.5, axis=0), device=dev)
hi = torch.tensor(np.percentile(Z[trn], 99.5, axis=0), device=dev)
PW = float((1 - y[fit].mean()) / y[fit].mean())
alt_te = te[cat[te] == "data alteration"]; nor_te = te[y[te] == 0]; alt_fit = fit[cat[fit] == "data alteration"]
x0 = Zt[alt_te]

def train_clf(Xt, yt, seed, epochs=40):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(34,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1)).to(dev)
    A = torch.tensor(Xt, device=dev); t = torch.tensor(yt, dtype=torch.float32, device=dev)
    opt = torch.optim.Adam(m.parameters(), 1e-3); pw = torch.tensor(PW, device=dev)
    for ep in range(epochs):
        perm = torch.randperm(len(A), device=dev)
        for i in range(0, len(A), 256):
            idx = perm[i:i+256]; opt.zero_grad()
            nn.functional.binary_cross_entropy_with_logits(m(A[idx]).squeeze(1), t[idx], pos_weight=pw).backward(); opt.step()
    m.eval()
    return m, (lambda T: torch.sigmoid(m(T).squeeze(-1)).detach())

def calibrate(fn): return float(np.percentile(fn(Zt[val[y[val] == 0]]).cpu().numpy(), 95))

def attack_es(fn, x0, eps, seed, gens=25, pop=16, sigma_div=2.0):      # guided (1+lambda)-ES
    g = torch.Generator(device=dev).manual_seed(seed); S, D = x0.shape
    lo_s = torch.minimum(lo, x0); hi_s = torch.maximum(hi, x0); best_x, best_s = x0.clone(), fn(x0)
    for _ in range(gens):
        cand = best_x[:, None, :] + torch.randn(S, pop, D, device=dev, generator=g) * (eps / sigma_div)
        cand = x0[:, None, :] + (cand - x0[:, None, :]).clamp(-eps, eps)
        cand = torch.minimum(torch.maximum(cand, lo_s[:, None, :]), hi_s[:, None, :])
        sc = fn(cand.reshape(-1, D)).reshape(S, pop); mn, ai = sc.min(1); upd = mn < best_s
        best_s = torch.where(upd, mn, best_s); best_x = torch.where(upd[:, None], cand[torch.arange(S), ai], best_x)
    return best_s, best_x

def attack_random(fn, x0, eps, seed, gens=25, pop=16):
    g = torch.Generator(device=dev).manual_seed(seed); S, D = x0.shape
    lo_s = torch.minimum(lo, x0); hi_s = torch.maximum(hi, x0); best_x, best_s = x0.clone(), fn(x0)
    for _ in range(gens):
        cand = x0[:, None, :] + (torch.rand(S, pop, D, device=dev, generator=g) * 2 - 1) * eps
        cand = torch.minimum(torch.maximum(cand, lo_s[:, None, :]), hi_s[:, None, :])
        sc = fn(cand.reshape(-1, D)).reshape(S, pop); mn, ai = sc.min(1); upd = mn < best_s
        best_s = torch.where(upd, mn, best_s); best_x = torch.where(upd[:, None], cand[torch.arange(S), ai], best_x)
    return best_s, best_x

def attack_pgd(model, x0, eps, seed, steps=50, restarts=2):             # white-box, L-inf, on the logit
    g = torch.Generator(device=dev).manual_seed(seed)
    lo_s = torch.minimum(lo, x0); hi_s = torch.maximum(hi, x0); alpha = eps / 10
    best_x = x0.clone(); best_l = model(x0).squeeze(-1).detach()
    for r in range(restarts):
        x = x0 + (torch.rand(x0.shape, device=dev, generator=g) * 2 - 1) * eps * float(r > 0)
        x = torch.minimum(torch.maximum(x, lo_s), hi_s)
        for _ in range(steps):
            x = x.detach().requires_grad_(True)
            model(x).squeeze(-1).sum().backward()
            with torch.no_grad():
                x = x - alpha * x.grad.sign()
                x = x0 + (x - x0).clamp(-eps, eps); x = torch.minimum(torch.maximum(x, lo_s), hi_s)
                l = model(x).squeeze(-1); upd = l < best_l
                best_l = torch.where(upd, l, best_l); best_x = torch.where(upd[:, None], x, best_x)
    return torch.sigmoid(best_l), best_x

EPS = [0.25, 0.5, 1.0]; rows = []; t0 = time.time()
for seed in range(3):
    # ---- hardening exactly as in 8d ----
    m0, f0 = train_clf(Z[fit], y[fit], seed); thr0 = calibrate(f0)
    adv = np.concatenate([attack_es(f0, Zt[alt_fit], e, 200 + seed*10 + k)[1].cpu().numpy() for k, e in enumerate(EPS)])
    m1, f1 = train_clf(np.vstack([Z[fit], adv]), np.r_[y[fit], np.ones(len(adv), dtype=y.dtype)], seed); thr1 = calibrate(f1)
    adv2 = np.concatenate([attack_es(f1, Zt[alt_fit], e, 300 + seed*10 + k)[1].cpu().numpy() for k, e in enumerate(EPS)])
    m2, f2 = train_clf(np.vstack([Z[fit], adv, adv2]), np.r_[y[fit], np.ones(len(adv)+len(adv2), dtype=y.dtype)], seed); thr2 = calibrate(f2)
    sur_m, sur_f = train_clf(Z[fit], y[fit], seed + 100)             # surrogate: clean data, different seed
    for rname, (m, f, thr) in {"round 0 (clean)": (m0, f0, thr0), "round 1": (m1, f1, thr1), "round 2": (m2, f2, thr2)}.items():
        det0 = f(x0) > thr; n0 = max(int(det0.sum()), 1)
        fpr = float((f(Zt[nor_te]) > thr).float().mean())
        for eps in EPS:
            res = {
              "ES (training attacker)":          attack_es(f, x0, eps, 1000 + seed)[0],
              "Random search":                   attack_random(f, x0, eps, 2000 + seed)[0],
              "ES 5x budget, sigma=eps/3":       attack_es(f, x0, eps, 5000 + seed, gens=125, sigma_div=3.0)[0],
              "PGD white-box":                   attack_pgd(m, x0, eps, 7000 + seed)[0],
              "PGD transfer (surrogate)":        f(attack_pgd(sur_m, x0, eps, 8000 + seed)[1]),
            }
            for name, s1 in res.items():
                rows.append(dict(seed=seed, round=rname, eps=eps, attacker=name,
                                 evasion=float((det0 & (s1 <= thr)).sum() / n0), test_fpr=fpr))
    print("seed", seed, "done", round(time.time() - t0), "s", flush=True)

R = pd.DataFrame(rows); R.to_csv(f"{OUT}/redteam_9b.csv", index=False)
agg = R.groupby(["attacker", "eps", "round"]).evasion.agg(["mean", "std"]).round(3)
print(agg.unstack("round").to_string())
print("\nTest FPR by round:"); print(R.groupby("round").test_fpr.agg(["mean", "std"]).round(3).to_string())

Mounted at /content/drive
seed 0 done 19 s
seed 1 done 30 s
seed 2 done 41 s
                                          mean                             std                
round                          round 0 (clean) round 1 round 2 round 0 (clean) round 1 round 2
attacker                  eps                                                                 
ES (training attacker)    0.25           0.057   0.005   0.000           0.056   0.004   0.000
                          0.50           0.394   0.129   0.001           0.199   0.110   0.002
                          1.00           0.875   0.784   0.344           0.044   0.070   0.226
ES 5x budget, sigma=eps/3 0.25           0.071   0.005   0.000           0.076   0.004   0.000
                          0.50           0.464   0.172   0.003           0.220   0.148   0.004
                          1.00           0.883   0.812   0.452           0.048   0.058   0.220
PGD transfer (surrogate)  0.25           0.062   0.003   0.000      

In [2]:
# ===== 9b-2: does robustness survive an exactly matched 5% test FPR? (self-contained) =====
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from google.colab import drive; drive.mount("/content/drive")
P   = "/content/drive/MyDrive/AI-DTSec/data/wustl_ehms/wustl-ehms-2020_with_attacks_categories.csv"
OUT = "/content/drive/MyDrive/AI-DTSec/results/redteam"
dev = "cuda" if torch.cuda.is_available() else "cpu"
b = np.load(f"{OUT}/baseline.npz", allow_pickle=True)
mu, sd, te, tr, cols = b["mu"], b["sd"], b["te"], b["tr"], list(b["cols"])
df = pd.read_csv(P); df.columns = [c.strip() for c in df.columns]
y = df["Label"].astype(int).values; cat = df["Attack Category"].astype(str).str.lower().values
X = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0).values.astype(np.float32)
Z = ((X - mu) / sd).astype(np.float32); Zt = torch.tensor(Z, device=dev)
tr = np.sort(tr); nval = len(tr) // 5
val, fit = tr[-nval:], tr[:-nval]; fit = fit[fit < val.min() - 100]; trn = fit[y[fit] == 0]
lo = torch.tensor(np.percentile(Z[trn], 0.5, axis=0), device=dev); hi = torch.tensor(np.percentile(Z[trn], 99.5, axis=0), device=dev)
PW = float((1 - y[fit].mean()) / y[fit].mean())
alt_te = te[cat[te] == "data alteration"]; nor_te = te[y[te] == 0]; alt_fit = fit[cat[fit] == "data alteration"]
x0 = Zt[alt_te]

def train_clf(Xt, yt, seed, epochs=40):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(34,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1)).to(dev)
    A = torch.tensor(Xt, device=dev); t = torch.tensor(yt, dtype=torch.float32, device=dev)
    opt = torch.optim.Adam(m.parameters(), 1e-3); pw = torch.tensor(PW, device=dev)
    for ep in range(epochs):
        perm = torch.randperm(len(A), device=dev)
        for i in range(0, len(A), 256):
            idx = perm[i:i+256]; opt.zero_grad()
            nn.functional.binary_cross_entropy_with_logits(m(A[idx]).squeeze(1), t[idx], pos_weight=pw).backward(); opt.step()
    m.eval()
    return m, (lambda T: torch.sigmoid(m(T).squeeze(-1)).detach())

def attack_es(fn, x0, eps, seed, gens=25, pop=16, sigma_div=2.0):
    g = torch.Generator(device=dev).manual_seed(seed); S, D = x0.shape
    lo_s = torch.minimum(lo, x0); hi_s = torch.maximum(hi, x0); best_x, best_s = x0.clone(), fn(x0)
    for _ in range(gens):
        cand = best_x[:, None, :] + torch.randn(S, pop, D, device=dev, generator=g) * (eps / sigma_div)
        cand = x0[:, None, :] + (cand - x0[:, None, :]).clamp(-eps, eps)
        cand = torch.minimum(torch.maximum(cand, lo_s[:, None, :]), hi_s[:, None, :])
        sc = fn(cand.reshape(-1, D)).reshape(S, pop); mn, ai = sc.min(1); upd = mn < best_s
        best_s = torch.where(upd, mn, best_s); best_x = torch.where(upd[:, None], cand[torch.arange(S), ai], best_x)
    return best_s, best_x

def attack_pgd(model, x0, eps, seed, steps=50, restarts=2):
    g = torch.Generator(device=dev).manual_seed(seed)
    lo_s = torch.minimum(lo, x0); hi_s = torch.maximum(hi, x0); alpha = eps / 10
    best_x = x0.clone(); best_l = model(x0).squeeze(-1).detach()
    for r in range(restarts):
        x = x0 + (torch.rand(x0.shape, device=dev, generator=g) * 2 - 1) * eps * float(r > 0)
        x = torch.minimum(torch.maximum(x, lo_s), hi_s)
        for _ in range(steps):
            x = x.detach().requires_grad_(True); model(x).squeeze(-1).sum().backward()
            with torch.no_grad():
                x = x - alpha * x.grad.sign(); x = x0 + (x - x0).clamp(-eps, eps); x = torch.minimum(torch.maximum(x, lo_s), hi_s)
                l = model(x).squeeze(-1); upd = l < best_l; best_l = torch.where(upd, l, best_l); best_x = torch.where(upd[:, None], x, best_x)
    return torch.sigmoid(best_l), best_x

q95 = lambda a: float(np.percentile(a.cpu().numpy(), 95))
EPS = [0.25, 0.5, 1.0]; rows = []; ops = []; t0 = time.time()
for seed in range(3):
    m0, f0 = train_clf(Z[fit], y[fit], seed)
    adv = np.concatenate([attack_es(f0, Zt[alt_fit], e, 200 + seed*10 + k)[1].cpu().numpy() for k, e in enumerate(EPS)])
    m1, f1 = train_clf(np.vstack([Z[fit], adv]), np.r_[y[fit], np.ones(len(adv), dtype=y.dtype)], seed)
    adv2 = np.concatenate([attack_es(f1, Zt[alt_fit], e, 300 + seed*10 + k)[1].cpu().numpy() for k, e in enumerate(EPS)])
    m2, f2 = train_clf(np.vstack([Z[fit], adv, adv2]), np.r_[y[fit], np.ones(len(adv)+len(adv2), dtype=y.dtype)], seed)
    sur_m, _ = train_clf(Z[fit], y[fit], seed + 100)
    for rname, (m, f) in {"round 0 (clean)": (m0, f0), "round 1": (m1, f1), "round 2": (m2, f2)}.items():
        thr_val  = q95(f(Zt[val[y[val] == 0]]))          # validation-calibrated (as in 8d / 9b)
        thr_test = q95(f(Zt[nor_te]))                    # oracle: exactly 5% FPR on the test normals
        for tname, thr in (("validation-calibrated", thr_val), ("test-matched 5% FPR", thr_test)):
            ops.append(dict(seed=seed, round=rname, threshold=tname,
                            test_fpr=float((f(Zt[nor_te]) > thr).float().mean()),
                            recall_alt=float((f(x0) > thr).float().mean())))
        for eps in EPS:
            res = {"ES (training attacker)": attack_es(f, x0, eps, 1000 + seed)[0],
                   "ES 5x budget, sigma=eps/3": attack_es(f, x0, eps, 5000 + seed, gens=125, sigma_div=3.0)[0],
                   "PGD white-box": attack_pgd(m, x0, eps, 7000 + seed)[0],
                   "PGD transfer (surrogate)": f(attack_pgd(sur_m, x0, eps, 8000 + seed)[1])}
            for tname, thr in (("validation-calibrated", thr_val), ("test-matched 5% FPR", thr_test)):
                det0 = f(x0) > thr; n0 = max(int(det0.sum()), 1)
                for name, s1 in res.items():
                    rows.append(dict(seed=seed, round=rname, eps=eps, attacker=name, threshold=tname,
                                     evasion=float((det0 & (s1 <= thr)).sum() / n0)))
    print("seed", seed, "done", round(time.time() - t0), "s", flush=True)

R = pd.DataFrame(rows); R.to_csv(f"{OUT}/redteam_9b2.csv", index=False)
O = pd.DataFrame(ops); O.to_csv(f"{OUT}/redteam_9b2_ops.csv", index=False)
print("\nOPERATING POINTS (mean over seeds)")
print(O.groupby(["threshold", "round"])[["test_fpr", "recall_alt"]].agg(["mean", "std"]).round(3).to_string())
print("\nEVASION (mean, std over seeds)")
print(R.groupby(["attacker", "eps", "threshold", "round"]).evasion.agg(["mean", "std"]).round(3)
        .unstack(["threshold", "round"]).to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
seed 0 done 10 s
seed 1 done 21 s
seed 2 done 31 s

OPERATING POINTS (mean over seeds)
                                      test_fpr        recall_alt       
                                          mean    std       mean    std
threshold             round                                            
test-matched 5% FPR   round 0 (clean)    0.050  0.000      0.990  0.004
                      round 1            0.050  0.000      0.994  0.004
                      round 2            0.050  0.000      0.991  0.004
validation-calibrated round 0 (clean)    0.071  0.010      0.994  0.004
                      round 1            0.080  0.084      0.994  0.004
                      round 2            0.100  0.032      0.991  0.004

EVASION (mean, std over seeds)
                                              mean                                                      

In [3]:
# ===== 9c: federated hardening across 4 simulated hospitals (self-contained) =====
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from google.colab import drive; drive.mount("/content/drive")
from sklearn.metrics import roc_auc_score
P   = "/content/drive/MyDrive/AI-DTSec/data/wustl_ehms/wustl-ehms-2020_with_attacks_categories.csv"
OUT = "/content/drive/MyDrive/AI-DTSec/results/redteam"
dev = "cuda" if torch.cuda.is_available() else "cpu"
b = np.load(f"{OUT}/baseline.npz", allow_pickle=True)
mu, sd, te, tr, cols = b["mu"], b["sd"], b["te"], b["tr"], list(b["cols"])
df = pd.read_csv(P); df.columns = [c.strip() for c in df.columns]
y = df["Label"].astype(int).values; cat = df["Attack Category"].astype(str).str.lower().values
X = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0).values.astype(np.float32)
Z = ((X - mu) / sd).astype(np.float32); Zt = torch.tensor(Z, device=dev)
tr = np.sort(tr); nval = len(tr) // 5
val, fit = tr[-nval:], tr[:-nval]; fit = fit[fit < val.min() - 100]; trn = fit[y[fit] == 0]
lo = torch.tensor(np.percentile(Z[trn], 0.5, axis=0), device=dev); hi = torch.tensor(np.percentile(Z[trn], 99.5, axis=0), device=dev)
PW = float((1 - y[fit].mean()) / y[fit].mean())
alt_te = te[cat[te] == "data alteration"]; spo_te = te[cat[te] == "spoofing"]; nor_te = te[y[te] == 0]
x0 = Zt[alt_te]; xsp = Zt[spo_te]; xval_n = Zt[val[y[val] == 0]]; xnor = Zt[nor_te]
EPS = [0.25, 0.5, 1.0]; N_CLIENTS = 4; ROUNDS = 40

def make_model():
    return nn.Sequential(nn.Linear(34,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1)).to(dev)
def local_train(m, A, t, epochs):
    opt = torch.optim.Adam(m.parameters(), 1e-3); pw = torch.tensor(PW, device=dev); m.train()
    for ep in range(epochs):
        perm = torch.randperm(len(A), device=dev)
        for i in range(0, len(A), 256):
            idx = perm[i:i+256]; opt.zero_grad()
            nn.functional.binary_cross_entropy_with_logits(m(A[idx]).squeeze(1), t[idx], pos_weight=pw).backward(); opt.step()
def train_central(Xc, yc, seed, epochs=40):
    torch.manual_seed(seed); m = make_model()
    local_train(m, torch.tensor(Xc, device=dev), torch.tensor(yc, dtype=torch.float32, device=dev), epochs); m.eval(); return m
def fedavg(clients, seed, rounds=ROUNDS):
    torch.manual_seed(seed); g = make_model()
    sizes = np.array([len(c["y"]) for c in clients], float); w = sizes / sizes.sum()
    data = [(torch.tensor(c["X"], device=dev), torch.tensor(c["y"], dtype=torch.float32, device=dev)) for c in clients]
    for r in range(rounds):
        states = []
        for A, t in data:
            m = make_model(); m.load_state_dict(g.state_dict()); local_train(m, A, t, 1); states.append(m.state_dict())
        g.load_state_dict({k: sum(float(w[i]) * states[i][k] for i in range(len(states))) for k in states[0]})
    g.eval(); return g
fn = lambda m: (lambda T: torch.sigmoid(m(T).squeeze(-1)).detach())

def attack_es(f, x0_, eps, seed, gens=25, pop=16, sigma_div=2.0):
    g = torch.Generator(device=dev).manual_seed(seed); S, D = x0_.shape
    lo_s = torch.minimum(lo, x0_); hi_s = torch.maximum(hi, x0_); best_x, best_s = x0_.clone(), f(x0_)
    for _ in range(gens):
        cand = best_x[:, None, :] + torch.randn(S, pop, D, device=dev, generator=g) * (eps / sigma_div)
        cand = x0_[:, None, :] + (cand - x0_[:, None, :]).clamp(-eps, eps)
        cand = torch.minimum(torch.maximum(cand, lo_s[:, None, :]), hi_s[:, None, :])
        sc = f(cand.reshape(-1, D)).reshape(S, pop); mn, ai = sc.min(1); upd = mn < best_s
        best_s = torch.where(upd, mn, best_s); best_x = torch.where(upd[:, None], cand[torch.arange(S), ai], best_x)
    return best_s, best_x
def attack_pgd(m, x0_, eps, seed, steps=50, restarts=2):
    g = torch.Generator(device=dev).manual_seed(seed)
    lo_s = torch.minimum(lo, x0_); hi_s = torch.maximum(hi, x0_); alpha = eps / 10
    best_x = x0_.clone(); best_l = m(x0_).squeeze(-1).detach()
    for r in range(restarts):
        x = x0_ + (torch.rand(x0_.shape, device=dev, generator=g) * 2 - 1) * eps * float(r > 0)
        x = torch.minimum(torch.maximum(x, lo_s), hi_s)
        for _ in range(steps):
            x = x.detach().requires_grad_(True); m(x).squeeze(-1).sum().backward()
            with torch.no_grad():
                x = x - alpha * x.grad.sign(); x = x0_ + (x - x0_).clamp(-eps, eps); x = torch.minimum(torch.maximum(x, lo_s), hi_s)
                l = m(x).squeeze(-1); upd = l < best_l; best_l = torch.where(upd, l, best_l); best_x = torch.where(upd[:, None], x, best_x)
    return torch.sigmoid(best_l), best_x

q95 = lambda a: float(np.percentile(a.cpu().numpy(), 95))
def evaluate(models, seed):
    outs = []
    for m in models:
        f = fn(m); thr = {"v": q95(f(xval_n)), "t": q95(f(xnor))}      # v = validation-calibrated, t = test-matched oracle
        r = dict(AUC=roc_auc_score(y[te], f(Zt[te]).cpu().numpy()), FPR=float((f(xnor) > thr["v"]).float().mean()),
                 rec_alt=float((f(x0) > thr["v"]).float().mean()), rec_spoof=float((f(xsp) > thr["v"]).float().mean()))
        for eps in (0.5, 1.0):
            att = {"ES": attack_es(f, x0, eps, 1000 + seed)[0], "PGD": attack_pgd(m, x0, eps, 7000 + seed)[0]}
            for tn in ("v", "t"):
                det0 = f(x0) > thr[tn]; n0 = max(int(det0.sum()), 1)
                for an, s1 in att.items(): r[f"{an}@{eps}[{tn}]"] = float((det0 & (s1 <= thr[tn])).sum() / n0)
        outs.append(r)
    return {k: float(np.mean([o[k] for o in outs])) for k in outs[0]}

def run_setting(mode, seed):
    if mode == "central": parts = [fit]
    else: parts = np.array_split(fit, N_CLIENTS)
    clients = [dict(X=Z[p].copy(), y=y[p].copy(), alt=Zt[p[cat[p] == "data alteration"]]) for p in parts]
    train = (lambda cl: [fedavg(cl, seed)]) if mode == "fed" else (lambda cl: [train_central(c["X"], c["y"], seed) for c in cl])
    models = train(clients); res = {"round 0": evaluate(models, seed)}
    for rnd in (1, 2):
        for ci, c in enumerate(clients):
            if len(c["alt"]) == 0: continue
            m = models[0] if mode == "fed" else models[ci]; f = fn(m)
            advs = [attack_es(f, c["alt"], e, 200 + seed*10 + rnd*100 + ci*1000 + k)[1].cpu().numpy() for k, e in enumerate(EPS)]
            c["X"] = np.vstack([c["X"]] + advs); c["y"] = np.r_[c["y"], np.ones(sum(len(a) for a in advs), dtype=c["y"].dtype)]
        models = train(clients); res[f"round {rnd}"] = evaluate(models, seed)
    return res

parts = np.array_split(fit, N_CLIENTS)
print("HOSPITALS (fitting data):")
for i, p in enumerate(parts):
    print(f"  hospital {i+1}: n={len(p)} normal={(y[p]==0).sum()} alteration={(cat[p]=='data alteration').sum()} spoofing={(cat[p]=='spoofing').sum()}")
rows = []; t0 = time.time()
for seed in range(3):
    for mode in ("local", "central", "fed"):
        for rname, r in run_setting(mode, seed).items(): rows.append(dict(seed=seed, setting=mode, round=rname, **r))
        print("seed", seed, mode, "done", round(time.time() - t0), "s", flush=True)
R = pd.DataFrame(rows); R.to_csv(f"{OUT}/redteam_9c.csv", index=False)
g = R.groupby(["setting", "round"]); order = ["local", "central", "fed"]
fmt = lambda c: (g[c].mean().round(3).astype(str) + " ± " + g[c].std().round(3).astype(str))
print("\nDETECTION AND OPERATING POINT (mean ± SD over seeds)")
print(pd.DataFrame({c: fmt(c) for c in ["AUC", "FPR", "rec_alt", "rec_spoof"]}).reindex(order, level=0).to_string())
for tn, title in (("t", "EVASION at test-matched 5% FPR threshold"), ("v", "EVASION at validation-calibrated threshold")):
    print(f"\n{title}")
    print(pd.DataFrame({c: fmt(c) for c in [f"ES@0.5[{tn}]", f"PGD@0.5[{tn}]", f"ES@1.0[{tn}]", f"PGD@1.0[{tn}]"]}).reindex(order, level=0).to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HOSPITALS (fitting data):
  hospital 1: n=2243 normal=1951 alteration=43 spoofing=249
  hospital 2: n=2243 normal=1903 alteration=163 spoofing=177
  hospital 3: n=2243 normal=1901 alteration=166 spoofing=176
  hospital 4: n=2242 normal=2021 alteration=127 spoofing=94
seed 0 local done 9 s
seed 0 central done 15 s
seed 0 fed done 22 s
seed 1 local done 29 s
seed 1 central done 36 s
seed 1 fed done 42 s
seed 2 local done 50 s
seed 2 central done 57 s
seed 2 fed done 63 s

DETECTION AND OPERATING POINT (mean ± SD over seeds)
                           AUC            FPR        rec_alt      rec_spoof
setting round                                                              
local   round 0  0.834 ± 0.004  0.087 ± 0.007  0.972 ± 0.005  0.135 ± 0.014
        round 1  0.824 ± 0.006  0.131 ± 0.018  0.986 ± 0.001  0.191 ± 0.038
        round 2  0.826 ± 0.002   0.074 

In [4]:
# ===== 9c-2: why is FedAvg more robust? controls on clean models (self-contained) =====
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from google.colab import drive; drive.mount("/content/drive")
from sklearn.metrics import roc_auc_score
P   = "/content/drive/MyDrive/AI-DTSec/data/wustl_ehms/wustl-ehms-2020_with_attacks_categories.csv"
OUT = "/content/drive/MyDrive/AI-DTSec/results/redteam"
dev = "cuda" if torch.cuda.is_available() else "cpu"
b = np.load(f"{OUT}/baseline.npz", allow_pickle=True)
mu, sd, te, tr, cols = b["mu"], b["sd"], b["te"], b["tr"], list(b["cols"])
df = pd.read_csv(P); df.columns = [c.strip() for c in df.columns]
y = df["Label"].astype(int).values; cat = df["Attack Category"].astype(str).str.lower().values
X = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0).values.astype(np.float32)
Z = ((X - mu) / sd).astype(np.float32); Zt = torch.tensor(Z, device=dev)
tr = np.sort(tr); nval = len(tr) // 5
val, fit = tr[-nval:], tr[:-nval]; fit = fit[fit < val.min() - 100]; trn = fit[y[fit] == 0]
lo = torch.tensor(np.percentile(Z[trn], 0.5, axis=0), device=dev); hi = torch.tensor(np.percentile(Z[trn], 99.5, axis=0), device=dev)
PW = float((1 - y[fit].mean()) / y[fit].mean())
alt_te = te[cat[te] == "data alteration"]; spo_te = te[cat[te] == "spoofing"]; nor_te = te[y[te] == 0]
x0 = Zt[alt_te]; xsp = Zt[spo_te]; xval_n = Zt[val[y[val] == 0]]; xnor = Zt[nor_te]
N_CLIENTS = 4

def make_model():
    return nn.Sequential(nn.Linear(34,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1)).to(dev)
def local_train(m, A, t, epochs):
    opt = torch.optim.Adam(m.parameters(), 1e-3); pw = torch.tensor(PW, device=dev); m.train()
    for ep in range(epochs):
        perm = torch.randperm(len(A), device=dev)
        for i in range(0, len(A), 256):
            idx = perm[i:i+256]; opt.zero_grad()
            nn.functional.binary_cross_entropy_with_logits(m(A[idx]).squeeze(1), t[idx], pos_weight=pw).backward(); opt.step()
def train_central(Xc, yc, seed, epochs):
    torch.manual_seed(seed); m = make_model()
    local_train(m, torch.tensor(Xc, device=dev), torch.tensor(yc, dtype=torch.float32, device=dev), epochs); m.eval(); return m
def fedavg(parts, seed, rounds, E):
    torch.manual_seed(seed); g = make_model()
    data = [(torch.tensor(Z[p], device=dev), torch.tensor(y[p], dtype=torch.float32, device=dev)) for p in parts]
    w = np.array([len(p) for p in parts], float); w /= w.sum()
    for r in range(rounds):
        states = []
        for A, t in data:
            m = make_model(); m.load_state_dict(g.state_dict()); local_train(m, A, t, E); states.append(m.state_dict())
        g.load_state_dict({k: sum(float(w[i]) * states[i][k] for i in range(len(states))) for k in states[0]})
    g.eval(); return g
fn = lambda m: (lambda T: torch.sigmoid(m(T).squeeze(-1)).detach())

def attack_es(f, x0_, eps, seed, gens=25, pop=16):
    g = torch.Generator(device=dev).manual_seed(seed); S, D = x0_.shape
    lo_s = torch.minimum(lo, x0_); hi_s = torch.maximum(hi, x0_); best_x, best_s = x0_.clone(), f(x0_)
    for _ in range(gens):
        cand = best_x[:, None, :] + torch.randn(S, pop, D, device=dev, generator=g) * (eps / 2)
        cand = x0_[:, None, :] + (cand - x0_[:, None, :]).clamp(-eps, eps)
        cand = torch.minimum(torch.maximum(cand, lo_s[:, None, :]), hi_s[:, None, :])
        sc = f(cand.reshape(-1, D)).reshape(S, pop); mn, ai = sc.min(1); upd = mn < best_s
        best_s = torch.where(upd, mn, best_s); best_x = torch.where(upd[:, None], cand[torch.arange(S), ai], best_x)
    return best_s
def attack_pgd(m, x0_, eps, seed, steps=50, restarts=2):
    g = torch.Generator(device=dev).manual_seed(seed)
    lo_s = torch.minimum(lo, x0_); hi_s = torch.maximum(hi, x0_); alpha = eps / 10
    best_l = m(x0_).squeeze(-1).detach()
    for r in range(restarts):
        x = x0_ + (torch.rand(x0_.shape, device=dev, generator=g) * 2 - 1) * eps * float(r > 0)
        x = torch.minimum(torch.maximum(x, lo_s), hi_s)
        for _ in range(steps):
            x = x.detach().requires_grad_(True); m(x).squeeze(-1).sum().backward()
            with torch.no_grad():
                x = x - alpha * x.grad.sign(); x = x0_ + (x - x0_).clamp(-eps, eps); x = torch.minimum(torch.maximum(x, lo_s), hi_s)
                best_l = torch.minimum(best_l, m(x).squeeze(-1))
    return torch.sigmoid(best_l)

q95 = lambda a: float(np.percentile(a.cpu().numpy(), 95))
def evaluate(m, seed):
    f = fn(m); thr = q95(f(xnor))                                     # test-matched 5% FPR (oracle diagnostic)
    r = dict(AUC=roc_auc_score(y[te], f(Zt[te]).cpu().numpy()), rec_spoof=float((f(xsp) > thr).float().mean()),
             logit_abs=float(m(x0).abs().mean()))
    det0 = f(x0) > thr; n0 = max(int(det0.sum()), 1)
    for eps in (0.5, 1.0):
        r[f"ES@{eps}"]  = float((det0 & (attack_es(f, x0, eps, 1000 + seed) <= thr)).sum() / n0)
        r[f"PGD@{eps}"] = float((det0 & (attack_pgd(m, x0, eps, 7000 + seed) <= thr)).sum() / n0)
    return r

chrono = lambda seed: np.array_split(fit, N_CLIENTS)
iid    = lambda seed: np.array_split(np.random.default_rng(seed).permutation(fit), N_CLIENTS)
SETTINGS = {
 "central 40 epochs":          lambda s: train_central(Z[fit], y[fit], s, 40),
 "central 20 epochs":          lambda s: train_central(Z[fit], y[fit], s, 20),
 "central 10 epochs":          lambda s: train_central(Z[fit], y[fit], s, 10),
 "central 5 epochs":           lambda s: train_central(Z[fit], y[fit], s, 5),
 "FedAvg chrono, E=1, R=40":   lambda s: fedavg(chrono(s), s, 40, 1),
 "FedAvg iid,    E=1, R=40":   lambda s: fedavg(iid(s), s, 40, 1),
 "FedAvg chrono, E=5, R=8":    lambda s: fedavg(chrono(s), s, 8, 5),
 "FedAvg iid,    E=5, R=8":    lambda s: fedavg(iid(s), s, 8, 5),
}
rows = []; t0 = time.time()
for seed in range(6):
    for name, mk in SETTINGS.items():
        rows.append(dict(seed=seed, setting=name, **evaluate(mk(seed), seed)))
    print("seed", seed, "done", round(time.time() - t0), "s", flush=True)
R = pd.DataFrame(rows); R.to_csv(f"{OUT}/redteam_9c2.csv", index=False)
g = R.groupby("setting", sort=False)
fmt = lambda c: g[c].mean().round(3).astype(str) + " ± " + g[c].std().round(3).astype(str)
print("\nCLEAN MODELS, 6 seeds (mean ± SD); evasion at test-matched 5% FPR")
print(pd.DataFrame({c: fmt(c) for c in ["AUC", "rec_spoof", "logit_abs", "ES@0.5", "PGD@0.5", "ES@1.0", "PGD@1.0"]}).to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_476/1014178818.py:74: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  logit_abs=float(m(x0).abs().mean()))


seed 0 done 12 s
seed 1 done 24 s
seed 2 done 37 s
seed 3 done 48 s
seed 4 done 58 s
seed 5 done 69 s

CLEAN MODELS, 6 seeds (mean ± SD); evasion at test-matched 5% FPR
                                    AUC      rec_spoof                logit_abs         ES@0.5        PGD@0.5         ES@1.0        PGD@1.0
setting                                                                                                                                    
central 40 epochs         0.853 ± 0.005  0.115 ± 0.021  237637.586 ± 150803.363  0.428 ± 0.143   0.516 ± 0.15   0.873 ± 0.03   0.893 ± 0.03
central 20 epochs         0.857 ± 0.005   0.134 ± 0.01     176591.23 ± 170430.9  0.253 ± 0.106  0.341 ± 0.112  0.834 ± 0.022  0.856 ± 0.026
central 10 epochs         0.855 ± 0.002  0.134 ± 0.004  158801.623 ± 144321.786  0.069 ± 0.052  0.117 ± 0.069  0.773 ± 0.031   0.81 ± 0.016
central 5 epochs          0.834 ± 0.007  0.075 ± 0.014    80524.817 ± 35769.428  0.021 ± 0.011  0.035 ± 0.019  0.655 ± 0.054  0.734

In [5]:
# ===== 9c-3: hardening with matched training budgets: central-10, central-40, local-10, federated (self-contained) =====
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from google.colab import drive; drive.mount("/content/drive")
from sklearn.metrics import roc_auc_score
P   = "/content/drive/MyDrive/AI-DTSec/data/wustl_ehms/wustl-ehms-2020_with_attacks_categories.csv"
OUT = "/content/drive/MyDrive/AI-DTSec/results/redteam"
dev = "cuda" if torch.cuda.is_available() else "cpu"
b = np.load(f"{OUT}/baseline.npz", allow_pickle=True)
mu, sd, te, tr, cols = b["mu"], b["sd"], b["te"], b["tr"], list(b["cols"])
df = pd.read_csv(P); df.columns = [c.strip() for c in df.columns]
y = df["Label"].astype(int).values; cat = df["Attack Category"].astype(str).str.lower().values
X = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0).values.astype(np.float32)
Z = ((X - mu) / sd).astype(np.float32); Zt = torch.tensor(Z, device=dev)
tr = np.sort(tr); nval = len(tr) // 5
val, fit = tr[-nval:], tr[:-nval]; fit = fit[fit < val.min() - 100]; trn = fit[y[fit] == 0]
lo = torch.tensor(np.percentile(Z[trn], 0.5, axis=0), device=dev); hi = torch.tensor(np.percentile(Z[trn], 99.5, axis=0), device=dev)
PW = float((1 - y[fit].mean()) / y[fit].mean())
alt_te = te[cat[te] == "data alteration"]; spo_te = te[cat[te] == "spoofing"]; nor_te = te[y[te] == 0]
x0 = Zt[alt_te]; xsp = Zt[spo_te]; xnor = Zt[nor_te]
EPS = [0.25, 0.5, 1.0]; N_CLIENTS = 4

def make_model():
    return nn.Sequential(nn.Linear(34,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1)).to(dev)
def local_train(m, A, t, epochs):
    opt = torch.optim.Adam(m.parameters(), 1e-3); pw = torch.tensor(PW, device=dev); m.train()
    for ep in range(epochs):
        perm = torch.randperm(len(A), device=dev)
        for i in range(0, len(A), 256):
            idx = perm[i:i+256]; opt.zero_grad()
            nn.functional.binary_cross_entropy_with_logits(m(A[idx]).squeeze(1), t[idx], pos_weight=pw).backward(); opt.step()
def train_central(Xc, yc, seed, epochs):
    torch.manual_seed(seed); m = make_model()
    local_train(m, torch.tensor(Xc, device=dev), torch.tensor(yc, dtype=torch.float32, device=dev), epochs); m.eval(); return m
def fedavg(clients, seed, rounds=40, E=1):
    torch.manual_seed(seed); g = make_model()
    data = [(torch.tensor(c["X"], device=dev), torch.tensor(c["y"], dtype=torch.float32, device=dev)) for c in clients]
    w = np.array([len(c["y"]) for c in clients], float); w /= w.sum()
    for r in range(rounds):
        states = []
        for A, t in data:
            m = make_model(); m.load_state_dict(g.state_dict()); local_train(m, A, t, E); states.append(m.state_dict())
        g.load_state_dict({k: sum(float(w[i]) * states[i][k] for i in range(len(states))) for k in states[0]})
    g.eval(); return g
fn = lambda m: (lambda T: torch.sigmoid(m(T).squeeze(-1)).detach())

def attack_es(f, x0_, eps, seed, gens=25, pop=16, want_x=False):
    g = torch.Generator(device=dev).manual_seed(seed); S, D = x0_.shape
    lo_s = torch.minimum(lo, x0_); hi_s = torch.maximum(hi, x0_); best_x, best_s = x0_.clone(), f(x0_)
    for _ in range(gens):
        cand = best_x[:, None, :] + torch.randn(S, pop, D, device=dev, generator=g) * (eps / 2)
        cand = x0_[:, None, :] + (cand - x0_[:, None, :]).clamp(-eps, eps)
        cand = torch.minimum(torch.maximum(cand, lo_s[:, None, :]), hi_s[:, None, :])
        sc = f(cand.reshape(-1, D)).reshape(S, pop); mn, ai = sc.min(1); upd = mn < best_s
        best_s = torch.where(upd, mn, best_s); best_x = torch.where(upd[:, None], cand[torch.arange(S), ai], best_x)
    return (best_s, best_x) if want_x else best_s
def attack_pgd(m, x0_, eps, seed, steps=50, restarts=2):
    g = torch.Generator(device=dev).manual_seed(seed)
    lo_s = torch.minimum(lo, x0_); hi_s = torch.maximum(hi, x0_); alpha = eps / 10
    best_l = m(x0_).squeeze(-1).detach()
    for r in range(restarts):
        x = x0_ + (torch.rand(x0_.shape, device=dev, generator=g) * 2 - 1) * eps * float(r > 0)
        x = torch.minimum(torch.maximum(x, lo_s), hi_s)
        for _ in range(steps):
            x = x.detach().requires_grad_(True); m(x).squeeze(-1).sum().backward()
            with torch.no_grad():
                x = x - alpha * x.grad.sign(); x = x0_ + (x - x0_).clamp(-eps, eps); x = torch.minimum(torch.maximum(x, lo_s), hi_s)
                best_l = torch.minimum(best_l, m(x).squeeze(-1))
    return torch.sigmoid(best_l)

q95 = lambda a: float(np.percentile(a.cpu().numpy(), 95))
def evaluate(models, seed):
    outs = []
    for m in models:
        f = fn(m); thr = q95(f(xnor))                                  # test-matched 5% FPR (oracle diagnostic)
        r = dict(AUC=roc_auc_score(y[te], f(Zt[te]).cpu().numpy()), rec_alt=float((f(x0) > thr).float().mean()),
                 rec_spoof=float((f(xsp) > thr).float().mean()))
        det0 = f(x0) > thr; n0 = max(int(det0.sum()), 1)
        for eps in (0.5, 1.0):
            r[f"ES@{eps}"]  = float((det0 & (attack_es(f, x0, eps, 1000 + seed) <= thr)).sum() / n0)
            r[f"PGD@{eps}"] = float((det0 & (attack_pgd(m, x0, eps, 7000 + seed) <= thr)).sum() / n0)
        outs.append(r)
    return {k: float(np.mean([o[k] for o in outs])) for k in outs[0]}

def run_setting(mode, epochs, seed):
    parts = [fit] if mode == "central" else np.array_split(fit, N_CLIENTS)
    clients = [dict(X=Z[p].copy(), y=y[p].copy(), alt=Zt[p[cat[p] == "data alteration"]]) for p in parts]
    train = (lambda cl: [fedavg(cl, seed)]) if mode == "fed" else (lambda cl: [train_central(c["X"], c["y"], seed, epochs) for c in cl])
    models = train(clients); res = {"round 0": evaluate(models, seed)}
    for rnd in (1, 2):
        for ci, c in enumerate(clients):
            if len(c["alt"]) == 0: continue
            m = models[0] if mode == "fed" else models[ci]; f = fn(m)
            advs = [attack_es(f, c["alt"], e, 200 + seed*10 + rnd*100 + ci*1000 + k, want_x=True)[1].cpu().numpy() for k, e in enumerate(EPS)]
            c["X"] = np.vstack([c["X"]] + advs); c["y"] = np.r_[c["y"], np.ones(sum(len(a) for a in advs), dtype=c["y"].dtype)]
        models = train(clients); res[f"round {rnd}"] = evaluate(models, seed)
    return res

SETTINGS = [("central, 10 epochs", "central", 10), ("central, 40 epochs", "central", 40),
            ("local-only, 10 epochs", "local", 10), ("federated (FedAvg, E=1, R=40)", "fed", 1)]
rows = []; t0 = time.time()
for seed in range(6):
    for name, mode, ep in SETTINGS:
        for rname, r in run_setting(mode, ep, seed).items(): rows.append(dict(seed=seed, setting=name, round=rname, **r))
    print("seed", seed, "done", round(time.time() - t0), "s", flush=True)
R = pd.DataFrame(rows); R.to_csv(f"{OUT}/redteam_9c3.csv", index=False)
g = R.groupby(["setting", "round"], sort=False)
fmt = lambda c: g[c].mean().round(3).astype(str) + " ± " + g[c].std().round(3).astype(str)
print("\nHARDENING WITH MATCHED TRAINING BUDGETS (6 seeds, mean ± SD; evasion at test-matched 5% FPR)")
print(pd.DataFrame({c: fmt(c) for c in ["AUC", "rec_alt", "rec_spoof", "ES@0.5", "PGD@0.5", "ES@1.0", "PGD@1.0"]}).to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
seed 0 done 20 s
seed 1 done 40 s
seed 2 done 59 s
seed 3 done 79 s
seed 4 done 98 s
seed 5 done 118 s

HARDENING WITH MATCHED TRAINING BUDGETS (6 seeds, mean ± SD; evasion at test-matched 5% FPR)
                                                 AUC        rec_alt      rec_spoof         ES@0.5        PGD@0.5         ES@1.0        PGD@1.0
setting                       round                                                                                                           
central, 10 epochs            round 0  0.855 ± 0.002  0.999 ± 0.002  0.134 ± 0.004  0.069 ± 0.052  0.117 ± 0.069  0.773 ± 0.031   0.81 ± 0.016
                              round 1   0.847 ± 0.01  0.999 ± 0.002  0.088 ± 0.015      0.0 ± 0.0      0.0 ± 0.0  0.399 ± 0.131  0.501 ± 0.118
                              round 2  0.842 ± 0.009      1.0 ± 0.0  0.098 ± 0.028  0.001 ± 0.002  0.0